# 01 - Umbra Missingness Diagnostics Walkthrough

This notebook demonstrates how Umbra evaluates missingness mechanisms across synthetic MCAR, MAR, and MNAR regimes without pretending that observed data alone provides certainty.

In [ ]:
import numpy as np
import pandas as pd
from scripts.build_synthetic_benchmarks import generate_benchmark_battery
from umbra import littles_mcar_test, analyze_missingness_patterns, find_shadow_variables, diagnose_dataframe, explain_diagnostics

## 1. Load Synthetic Benchmark Battery
We generate synthetic datasets with known ground truth across missingness regimes.

In [ ]:
battery = generate_benchmark_battery(n_samples=2000, random_state=42)
mcar_data = battery['MCAR'].data_observed
mar_data = battery['MAR'].data_observed
mnar_data = battery['MNAR_MEDIUM'].data_observed

## 2. Little's MCAR Test
Tests whether missingness is consistent with MCAR globally. Rejection rules out MCAR, but cannot distinguish MAR from MNAR.

In [ ]:
res_mcar = littles_mcar_test(mcar_data)
print('MCAR Dataset Little\'s Test: p-value =', f'{res_mcar.p_value:.4f}', '| Rejected:', res_mcar.is_rejected)

res_mar = littles_mcar_test(mar_data)
print('MAR Dataset Little\'s Test: p-value =', f'{res_mar.p_value:.4e}', '| Rejected:', res_mar.is_rejected)

res_mnar = littles_mcar_test(mnar_data)
print('MNAR Dataset Little\'s Test: p-value =', f'{res_mnar.p_value:.4e}', '| Rejected:', res_mnar.is_rejected)

## 3. Covariate Shift Analysis
Compares observed covariate distributions between missing and observed rows.

In [ ]:
patterns = analyze_missingness_patterns(mar_data)
var_rep = patterns.variable_reports['income']
print('Significant Covariate Shifts:', var_rep.n_significant_shifts)
for covar, shift in var_rep.covariate_shifts.items():
    print(shift.summary())

## 4. Shadow Variable Candidate Finder
Surfaces potential instruments that correlate with missingness but not with the outcome directly.

In [ ]:
shadow_rep = find_shadow_variables(mnar_data, target_column='income')
print(shadow_rep.summary())

## 5. Comprehensive MNAR Risk Report
Combines Little's test, covariate shift, self-censoring heuristics, and domain priors.

In [ ]:
reports = diagnose_dataframe(mnar_data)
explain_diagnostics(reports)